# Laboratorio L10 - Metadati e reidentificazione

B.4 - Cybersicurezza e IA

- parte 1: metadati EXIF di una foto
- parte 2: reidentificazione di dati "anonimizzati" e k-anonimato
- parte 3: checklist sulle impostazioni di privacy

Tutti i dati sono fittizi. I file `L10_foto_esempio.jpg`, `L10_questionario_anonimizzato.csv` e `L10_iscritti_torneo.csv` devono trovarsi nella stessa cartella del notebook (in JupyterLite: trascinarli nel pannello dei file).

## 1. Leggere i metadati EXIF

- `Image.open(nome_file)`: apre un'immagine (libreria Pillow, importata come `PIL`)
- `img.getexif()`: restituisce i metadati EXIF come dizionario, con codici numerici come chiavi
- `ExifTags.TAGS`: dizionario che traduce i codici numerici in nomi leggibili
- `exif.get_ifd(0x8825)`: la sezione GPS dei metadati (`0x8825` è il codice esadecimale della sezione GPS)

In [ ]:
from PIL import Image, ExifTags

img = Image.open("L10_foto_esempio.jpg")
exif = img.getexif()
for codice, valore in exif.items():
    nome = ExifTags.TAGS.get(codice, codice)
    print(f"{nome:20} {valore}")

gps = exif.get_ifd(0x8825)
print()
print("sezione GPS:", gps)

## 2. Da gradi, minuti, secondi a gradi decimali

gradi decimali = gradi + minuti / 60 + secondi / 3600, con segno negativo per Sud (`S`) e Ovest (`W`).

In [ ]:
def in_decimali(valori, emisfero):
    gradi, minuti, secondi = (float(v) for v in valori)
    decimali = gradi + minuti / 60 + secondi / 3600
    return -decimali if emisfero in ("S", "W") else decimali

lat = in_decimali(gps[2], gps[1])
lon = in_decimali(gps[4], gps[3])
print(f"latitudine  {lat:.5f}")
print(f"longitudine {lon:.5f}")
print(f"su una mappa: https://www.openstreetmap.org/?mlat={lat:.5f}&mlon={lon:.5f}#map=18/{lat:.5f}/{lon:.5f}")

**Domanda**: aprire il link della mappa (se la rete è disponibile). Che cosa si scopre? Se la foto fosse stata scattata a casa, che cosa rivelerebbe?

Risposta:

## 3. Salvare una copia senza metadati (esercizio 2)

Si crea una nuova immagine con gli stessi pixel (`img.tobytes()` restituisce i dati dei pixel, `Image.frombytes` crea un'immagine da quei dati) e la si salva senza passare i metadati.

In [ ]:
pulita = Image.frombytes(img.mode, img.size, img.tobytes())   # stessi pixel, nessun metadato
pulita.save("L10_foto_senza_metadati.jpg", quality=88)

print("metadati della copia:", dict(Image.open("L10_foto_senza_metadati.jpg").getexif()))

Scaricare `L10_foto_senza_metadati.jpg` e verificarla con `Extract EXIF` in CyberChef.

## 4. Reidentificazione (esercizio 3)

Due insiemi di dati fittizi:

- `L10_questionario_anonimizzato.csv`: risposte a un questionario sul benessere, senza nome e cognome, ma con data di nascita, genere e CAP
- `L10_iscritti_torneo.csv`: elenco pubblico degli iscritti a un torneo sportivo, con nome, cognome, data di nascita, genere e CAP

Si usa la libreria pandas:

- `pd.read_csv(file, dtype=str)`: legge un file CSV in una tabella (DataFrame), trattando tutti i valori come testo (così il CAP `00118` non perde gli zeri iniziali)
- `tabella.head()`: prime righe
- `pd.merge(a, b, on=[colonne])`: unisce le righe delle due tabelle che hanno gli stessi valori nelle colonne indicate

In [ ]:
import pandas as pd

questionario = pd.read_csv("L10_questionario_anonimizzato.csv", dtype=str)
iscritti = pd.read_csv("L10_iscritti_torneo.csv", dtype=str)
print("righe del questionario:", len(questionario))
print("iscritti al torneo    :", len(iscritti))
questionario.head()

In [ ]:
QUASI_ID = ["data_nascita", "genere", "cap"]

incrocio = pd.merge(iscritti, questionario, on=QUASI_ID)
print("risposte attribuite a una persona con nome e cognome:", len(incrocio))
incrocio[["nome", "cognome", "squadra", "ore_sonno", "si_sente_sotto_pressione", "usa_social_di_notte"]]

**Domanda**: il questionario era stato presentato come anonimo. Lo era davvero? Quali colonne lo hanno reso reidentificabile?

Risposta:

## 5. k-anonimato e generalizzazione (esercizio 4)

Il k-anonimato di una tabella è la dimensione del gruppo più piccolo di righe che condividono gli stessi quasi-identificatori.

- `tabella.groupby(colonne).size()`: conta quante righe ci sono per ogni combinazione di valori
- `.min()`: il valore minimo
- `tabella["colonna"].str[:4]`: i primi 4 caratteri di ogni valore della colonna

In [ ]:
def k_anonimato(tabella, colonne):
    return tabella.groupby(colonne).size().min()

print("k prima della generalizzazione:", k_anonimato(questionario, QUASI_ID))

generalizzato = questionario.copy()
generalizzato["anno_nascita"] = generalizzato["data_nascita"].str[:4]
generalizzato["cap3"] = generalizzato["cap"].str[:3]
QUASI_ID_GEN = ["anno_nascita", "genere", "cap3"]
print("k dopo la generalizzazione  :", k_anonimato(generalizzato, QUASI_ID_GEN))

iscritti_gen = iscritti.copy()
iscritti_gen["anno_nascita"] = iscritti_gen["data_nascita"].str[:4]
iscritti_gen["cap3"] = iscritti_gen["cap"].str[:3]
incrocio_gen = pd.merge(iscritti_gen, generalizzato, on=QUASI_ID_GEN)
print("righe dell'incrocio dopo la generalizzazione:", len(incrocio_gen),
      "(ogni iscritto corrisponde a più risposte: non si sa quale sia la sua)")

**Domande**

- Quanto vale k prima e dopo? Che cosa significa k = 1?
- Dopo la generalizzazione l'incrocio restituisce molte righe per ogni iscritto: perché questo protegge le risposte?
- Quale informazione si perde generalizzando? Serviva davvero la data di nascita completa per un questionario sul benessere?

Risposte:

## 6. Checklist sulle impostazioni di privacy (esercizio 5)

Da compilare individualmente, guardando le impostazioni di un assistente di IA o di un social usato abitualmente. Non mostrare le proprie impostazioni alla classe.

| Domanda | Risposta |
|---|---|
| Le conversazioni o i contenuti vengono conservati? Per quanto tempo? | |
| Possono essere usati per addestrare modelli? Si può disattivare? | |
| Il servizio ha una "memoria" delle conversazioni precedenti? | |
| Esistono conversazioni o contenuti condivisi tramite link pubblico? | |
| Il servizio o l'app ha accesso alla posizione? È necessario? | |
| Ci sono dati di altre persone inseriti da me (foto, messaggi, documenti)? | |